In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
import pandas as pd

In [4]:
traits = pd.read_csv("mediterranean_species_traits_hypothetical.csv")

In [9]:
#Give each species an integer code
traits = traits.reset_index(drop=True)
traits["code"] = np.arange(len(traits))

In [18]:
species_names = traits["scientific_name"].to_numpy()
interception = traits["interception_mm"].to_numpy()
evap_wilting = traits["evap_wilting"].to_numpy()
evap_max = traits["evap_max"].to_numpy()
maxevap_pt = traits["maxevap_pt"].to_numpy()
wilting_pt = traits["wilting_pt"].to_numpy()
hygroscopic_pt = traits["hygroscopic_pt"].to_numpy()
shade_factor = traits["shade_factor"].to_numpy()

In [21]:
def make_forest(size, probs, rng=None):
    """
    Sample a forest grid of integer species codes.
    probs: lengh n_species, nonnegative, sums to 1
    """
    probs = np.array(probs, dtype=float)
    probs = probs/probs.sum()

    if rng is None:
        rng = np.random.default_rng()

    return rng.choice(np.arange(len(probs)), size=(size,size), p=probs)
    

In [23]:
#Daily temperature by season

def season_temp(day):
    """
    {Days} = {0,1,...,364}
    Constant temperature in each season, i.e., effects of temperature are being averaged along each season.
    """

    if day<90:         #winter
        return 8.0
        
    elif day<181:      #spring
        return 15.0

    elif day<273:      #summer
        return 24.0

    else:              #autumm
        return 14.0

In [24]:
#Species dependent evapotranspiration

def species_evapotransp(state,forest):
    """
    Piece-wise linear evapotranspiration as a function of soil-moisture
    state: 2D moisture array
    forest: 2D array of species codes
    """

    s = state
    ew = evap_wilting[forest]
    em = evap_max[forest]
    mp = maxevap_pt[forest]
    wp = wilting_pt[forest]
    hp = hygroscopic_pt[forest]

    et = np.zeros_like(s, dtype=float)    #create an array of zeros of same shape as s, to be updated with values of evapotranspiration

    #Below hygroscopic pt: no evapotranspiration

    #Boolean masks to define evapotranspiration piece-wise linearly

    m1 = (s > hp) & (s <= wp)
    m2 = (s> wp) & (s <= mp)
    m3 = (s > mp)

    et[m1] = ew[m1] * (s[m1] - hp[m1])/(wp[m1] - hp[m1])
    et[m2] = ew[m2] + ((em[m2] - ew[m2])* (s[m2] - wp[m2])/(mp[m2] - wp[m2]))
    et[m3] = em[m3]
                       
    

In [25]:
#Soil moisture evaporation depending on temperature and shade factors

def temp_evap(state, forest, day, base_evap=0.002, temp_sensitivity=0.0005):
    """
    Bare soil evaporation modified by seasonal temperature and shade where plants are present
    """

    temp = season_temp(day)
    climate_evap = base_evap + temp_sensitivity*temp

    shade = shade_factor[forest]
    shade_mult = 1 - shade
    new_state = state - climate_evap*shade_mult

    return np.clip(new_state,0,None)

In [26]:
#Rain input with interception

def rain(state,forest, lam=0.1, mean_depth=15.0):
    """
    One daily rainfall event process as Poisson process
    lam: Poisson rate parameter for rainfall recurrence
    mean_depth: exponential mean_depth if rainfall occurs
    """

    new_state = state.copy()
    rain_today = np.random.poisson(lam) > 0

    if rain_today:
        depth = np.random.exponential(mean_depth)
        eff_rain = np.maximum(depth - interception[forest],0.0)
        new_state = new_state + eff_rain

    return np.clip(new_state, 0, None)

In [27]:
#Diffusion

def diffuse (m, D=0.1):
    new_m = m.copy()
    for i in range(1, size-1):
        for j in range(1,size-1):
            neighbors = [m[i+1,j],m[i-1,j],m[i,j+1],m[i,j-1]]
            new_m[i,j]+= D*(np.sum(neighbors) - 4*m[i,j])
    return new_m

In [28]:
#Final moisture for one forest realization

def final_moisture(m, forest, sweeps=365, D=0.1, base_evap=0.002, temp_sensitivity=0.0005, lam=0.1, mean_depth=15.0):
    state = m.copy()
    state_day= []

    for day in range(sweeps):
        state = diffuse(state, D=D)
        state = species_evapotransp(state,forest)
        state = temp_evap(state, forest, day, base_evap = base_evap, temp_sensitivity = temp_sensitivity)
        state = rain_input(state,forest,lam=lam, mean_depth=mean_depth)
        average = np.mean(state)
        state_day.append(average)

    return state_day